# 01 — Data Ingestion and Chunking

Thin demo layer over `scripts/ingest.py`.  All configuration lives in `scripts/config.py`.

Shows:
- how the ingestion module is configured
- how PDF pages are extracted
- how chunks are created
- how `chunks.jsonl` is generated

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4


In [2]:
# All constants now come from config — no duplication across scripts.
from scripts.config import PDF_DIR, TEXT_DIR, CHUNKS_FILE, CHUNKING_METHOD
from scripts.ingest import extract_pdf_pages, build_chunks_from_pages, main as ingest_main

print("PDF_DIR       :", PDF_DIR)
print("TEXT_DIR      :", TEXT_DIR)
print("CHUNKS_FILE   :", CHUNKS_FILE)
print("CHUNKING_METHOD:", CHUNKING_METHOD)

PDF_DIR       : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\pdfs
TEXT_DIR      : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\texts
CHUNKS_FILE   : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\texts\chunks.jsonl
CHUNKING_METHOD: semantic


## Preview available PDFs

In [3]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
len(pdf_files), [p.name for p in pdf_files]

(15,
 ['01_Guest_Stay_and_Incidentals_Handbook.pdf',
  '02_Hosted_Travel_and_Accommodation_Policy.pdf',
  '03_Ground_Transportation,_Parking,_and_Personal_Vehicle_Policy.pdf',
  '04_Receipts,_Reimbursements,_and_Supporting_Documentation_Standard.pdf',
  '05_Guest_Safety,_Accessibility,_and_Reasonable_Comfort_Standard.pdf',
  '06_Conference,_Hosted_Dining,_and_Event_Billing_Policy.pdf',
  '07_Department_Approval_and_Chargeback_Workflow.pdf',
  '08_Corporate_Stay_FAQ_for_Sponsored_Guests.pdf',
  '09_Early_Arrival_and_Inventory_Management_Policy.pdf',
  '10_Room_Upgrade_and_Allocation_Guidelines.pdf',
  '11_Wi-Fi_and_Network_Access_Policy.pdf',
  '12_Noise_and_Disturbance_Enforcement_Policy.pdf',
  '13_Pet_and_Service_Animal_Compliance_Policy.pdf',
  '14_Billing_Dispute_Resolution_Procedure.pdf',
  '15_Security_and_After-Hours_Access_Policy.pdf'])

## Preview raw extracted pages from one PDF

In [4]:
sample_pdf = pdf_files[0]
pages = extract_pdf_pages(sample_pdf)
len(pages), pages[0]

(2,
 {'source': '01_Guest_Stay_and_Incidentals_Handbook.pdf',
  'page': 1,
  'text': 'Guest Stay and Incidentals Handbook\nArrival, Departure, and Holds\nStandard check-in begins at 3:00 PM with a government-issued photo ID required for each registering\nadult. Guests arriving ahead of schedule may request access beginning at 12:00 PM for $35, subject to\nhousekeeping release and same-day inventory pressure. Standard departure is 11:00 AM. A same-day\nextension to 2:00 PM may be approved for $50 when room recovery is not materially affected; later\ndepartures may be billed as an additional night.\nA payment card presented at arrival is used to secure room, tax, and a reasonable estimate of\nincidental charges. When the form of payment changes during the stay, front desk staff may request a\nrefreshed authorization. Luggage may be stored prior to room assignment and after departure; guests\nshould not place passports, cash, jewelry, or medications in held items.\nHouse Rules\n(cid:127) 

## Preview chunks from one PDF

In [5]:
sample_chunks = build_chunks_from_pages(pages)
len(sample_chunks), sample_chunks[:2]

(4,
 [{'chunk_id': '01_Guest_Stay_and_Incidentals_Handbook_p1_c1',
   'source': '01_Guest_Stay_and_Incidentals_Handbook.pdf',
   'page': 1,
   'title': '01 Guest Stay And Incidentals Handbook',
   'section': None,
   'text': 'Guest Stay and Incidentals Handbook\nArrival, Departure, and Holds\nStandard check-in begins at 3:00 PM with a government-issued photo ID required for each registering\nadult. Guests arriving ahead of schedule may request access beginning at 12:00 PM for $35, subject to\nhousekeeping release and same-day inventory pressure. Standard departure is 11:00 AM. A same-day\nextension to 2:00 PM may be approved for $50 when room recovery is not materially affected; later\ndepartures may be billed as an additional night.\nA payment card presented at arrival is used to secure room, tax, and a reasonable estimate of\nincidental charges. When the form of payment changes during the stay, front desk staff may request a\nrefreshed authorization. Luggage may be stored prior to ro

## Run the full ingestion pipeline

In [6]:
ingest_main()

Processed PDFs : 15
Extracted pages: 30
Saved chunks   : 33
Output file    : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\texts\chunks.jsonl
Chunking method: semantic


## Inspect the saved JSONL output

In [7]:
import json

rows = []
with CHUNKS_FILE.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rows.append(json.loads(line))
rows

[{'chunk_id': '01_Guest_Stay_and_Incidentals_Handbook_p1_c1',
  'source': '01_Guest_Stay_and_Incidentals_Handbook.pdf',
  'page': 1,
  'title': '01 Guest Stay And Incidentals Handbook',
  'section': None,
  'text': 'Guest Stay and Incidentals Handbook\nArrival, Departure, and Holds\nStandard check-in begins at 3:00 PM with a government-issued photo ID required for each registering\nadult. Guests arriving ahead of schedule may request access beginning at 12:00 PM for $35, subject to\nhousekeeping release and same-day inventory pressure. Standard departure is 11:00 AM. A same-day\nextension to 2:00 PM may be approved for $50 when room recovery is not materially affected; later\ndepartures may be billed as an additional night.\nA payment card presented at arrival is used to secure room, tax, and a reasonable estimate of\nincidental charges. When the form of payment changes during the stay, front desk staff may request a\nrefreshed authorization. Luggage may be stored prior to room assignm

In [8]:
#  BUILD BM25 INDEX

from scripts.build_index import main as build_index_main

build_index_main()

Loaded 33 chunks from C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\texts\chunks.jsonl
Building BM25 index...
Saved BM25 index → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\embeddings\bm25_index.pkl
Building Chroma dense index...
Saved Chroma collection → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\embeddings\chroma
Saved chunk metadata → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list - V2\aegis-rag-v4\data\embeddings\chunks_metadata.json

Done. Indexed 33 chunks.
Embedding model    : intfloat/e5-large-v2
Chroma collection  : hotel_chunks
